In [2]:
import requests
import pandas as pd
import numpy as np
import polars as pl

In [3]:
# Fetch help contacts data
help_contacts = requests.get("http://api.justgood.work/api/v2/help-contacts")
help_contacts_json = help_contacts.json()["data"]

In [4]:
# Convert help contacts json to a pandas Dataframe
hc_df = pd.DataFrame.from_records(help_contacts_json)

# view the first 2 records
hc_df.head()

,_id,deviceId,appId,locale,route,phoneNumber,organization,email,createdAt,updatedAt,__v,language
0,64754ce1bf8be67e27fb4122,jgw-vanilla-ABE918BE-7620-4C1E-8DC4-790E549562...,jgw-vanilla,GBR,GBR -> GBR,08000 121 700,,,2023-05-30T01:09:53.594Z,2023-05-30T01:09:53.594Z,0,NaN
1,6463a77057f25744aff36429,jgw-vanilla-afe14226d1d2386e-TJK -> GBR,jgw-vanilla,TJK,TJK -> GBR,08000 121 700,,,2023-05-16T15:55:28.363Z,2023-05-16T15:55:28.363Z,0,NaN
2,640ef2983f51d833637453f5,jgw-5158254C-33FD-4A66-81D3-1F9FD042AD26-madag...,jgw,madagascar,madagascar -> mauritius,None,In Mauritius: Ministry of Labour,mol@govmu.org,2023-03-13T09:53:28.125Z,2023-03-13T09:53:28.125Z,0,english
3,638add6d292f141036c92d08,jgw-undefined-bangladesh -> mauritius,jgw,bangladesh,bangladesh -> mauritius,+২৩০ ২০৭ ২৭১১,মরিশাসঃ শ্রম মন্ত্রণালয়,None,2022-12-03T05:23:57.694Z,2022-12-03T05:23:57.694Z,0,বাংলা
4,638add6d292f141036c92d07,jgw-undefined-bangladesh -> mauritius,jgw,bangladesh,bangladesh -> mauritius,+২৩০ ২০৭ ২৭১১,মরিশাসঃ শ্রম মন্ত্রণালয়,None,2022-12-03T05:23:57.494Z,2022-12-03T05:23:57.494Z,0,বাংলা


In [5]:
# generics
def count_unique(df, field):
    return df[field].nunique()

In [6]:
start_date = "2023-05-01" 
end_date = "2023-05-31"
mask = hc_df["createdAt"].between(start_date, end_date, inclusive="both")
hc_df = hc_df.loc[mask]

In [7]:
# phone numbers and emails by route    
def df_aggregate_by_group(df, result_fields, group_field):
    df[group_field] = df[group_field].replace("", "Not specified")
    
    aggregation_dict = {field: lambda values: values.count() for field in result_fields}
    grouped_data = df.groupby(group_field)[result_fields].agg(aggregation_dict)
    
    result = {
        "labels": grouped_data.index.values,
    }
    
    for field in result_fields:
        result[field] = grouped_data[field].values
    
    return result

In [8]:
def dictionary_from_two_lists(labels, values):
    """Creates a dictionary from two lists

    Args:
        labels (_list_): to be used as keys
        values (_list_): to be used as values

    Returns:
        _type_: returns a dictionary 
    """
    new_dict = {}

    for label, value in zip(labels, values):
        new_dict[label] = value

    return new_dict


_df = hc_df
emails_and_phonenumbers_clicks_by_route = df_aggregate_by_group(
    _df,
    result_fields=["phoneNumber", "email"],
    group_field="route"
)
emails_and_phonenumbers_clicks_by_org = df_aggregate_by_group(
    _df,
    result_fields=["phoneNumber", "email"],
    group_field="organization"
)
phone_numbers_by_route = dictionary_from_two_lists(
    emails_and_phonenumbers_clicks_by_route.get("labels"),
    emails_and_phonenumbers_clicks_by_route.get("phoneNumber")
)
emails_by_route = dictionary_from_two_lists(
    emails_and_phonenumbers_clicks_by_route.get("labels"),
    emails_and_phonenumbers_clicks_by_route.get("email")
)

phone_numbers_by_org = dictionary_from_two_lists(
    emails_and_phonenumbers_clicks_by_org.get("labels"),
    emails_and_phonenumbers_clicks_by_org.get("phoneNumber")
)
emails_by_org = dictionary_from_two_lists(
    emails_and_phonenumbers_clicks_by_org.get("labels"),
    emails_and_phonenumbers_clicks_by_org.get("email")
)
print({
    "phone numbers clicks by route": phone_numbers_by_route, 
    "emails clicks by route": emails_by_route,
    # "phone_numbers_by_org":phone_numbers_by_org,
    # "emails_by_org":emails_by_org
})
phone_numbers_by_route, emails_by_route

({'GBR -> GBR': 1, 'TJK -> GBR': 1}, {'GBR -> GBR': 1, 'TJK -> GBR': 1})